# Statistical Arbitrage Research & Backtesting Pipeline

**Pair studied:** TXN–GS  
**Data:** Daily adjusted close prices, 2021–2025  
**Design:** 2021–2023 training → 2024 validation → 2025 final out-of-sample test

This notebook builds a simple pairs-trading research pipeline using Engle–Granger
cointegration screening, OLS residual spreads, ADF diagnostics, mean-reversion
half-life estimation, Z-score trading rules, lagged execution and turnover-based
transaction costs.

The purpose is to evaluate whether an apparently promising mean-reversion
relationship survives chronological validation and unseen out-of-sample testing.
It is an educational research project, not investment advice.


## 1. Imports


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

from itertools import combinations
from statsmodels.tsa.stattools import coint, adfuller
from statsmodels.stats.multitest import multipletests


## 2. Data universe and chronological split

The final research universe contains 20 liquid U.S. equities across technology,
semiconductors, financials, payments and consumer names. This produces
$\binom{20}{2}=190$ candidate pairs.

The split is strictly chronological:

- **Training:** 2021–2023
- **Validation:** 2024
- **Final out-of-sample test:** 2025


In [ ]:
tickers = [
    "AAPL", "MSFT", "GOOGL", "META",
    "AMZN", "NVDA", "ORCL", "CRM",
    "AMD", "QCOM", "AVGO", "TXN",
    "JPM", "BAC", "GS", "MS",
    "V", "MA", "WMT", "COST"
]

raw = yf.download(
    tickers,
    start="2021-01-01",
    end="2026-01-01",
    auto_adjust=False,
    progress=False
)

prices = raw["Adj Close"].dropna()

train = prices.loc["2021-01-01":"2023-12-31"].copy()
validation = prices.loc["2024-01-01":"2024-12-31"].copy()
test = prices.loc["2025-01-01":"2025-12-31"].copy()

print("Stocks:", len(tickers))
print("Possible pairs:", len(tickers) * (len(tickers) - 1) // 2)
print("Dataset shape:", prices.shape)
print("Training observations:", len(train))
print("Validation observations:", len(validation))
print("Test observations:", len(test))


## 3. Training-only cointegration screen

All pair selection is performed using the **training period only**. Engle–Granger
cointegration p-values are calculated for all 190 pairs.

Because testing many pairs increases false-positive risk, the notebook also applies
a Benjamini–Hochberg false-discovery-rate (FDR) correction. No pair survives the
5% FDR threshold in this sample, so the selected pair should be treated as an
**exploratory candidate**, not as statistically robust evidence of cointegration
after multiple-testing adjustment.


In [ ]:
cointegration_results = []

for asset_a, asset_b in combinations(tickers, 2):
    y = train[asset_a]
    x = train[asset_b]

    test_stat, p_value, _ = coint(y, x)

    cointegration_results.append({
        "Asset_A": asset_a,
        "Asset_B": asset_b,
        "Pair": f"{asset_a}-{asset_b}",
        "Test_Statistic": test_stat,
        "P_Value": p_value
    })

cointegration_results = (
    pd.DataFrame(cointegration_results)
    .sort_values("P_Value")
    .reset_index(drop=True)
)

reject_fdr, corrected_pvalues, _, _ = multipletests(
    cointegration_results["P_Value"],
    alpha=0.05,
    method="fdr_bh"
)

cointegration_results["FDR_P_Value"] = corrected_pvalues
cointegration_results["Pass_FDR_5pct"] = reject_fdr

raw_candidates = cointegration_results[
    cointegration_results["P_Value"] < 0.05
].copy()

print("Raw p < 5% pairs:", len(raw_candidates))
print("Pairs surviving 5% FDR:",
      cointegration_results["Pass_FDR_5pct"].sum())

cointegration_results.head(12)


## 4. Residual diagnostics and mean-reversion half-life

For each raw training-period candidate, an OLS regression estimates

\[
A_t = \alpha + \beta B_t + \varepsilon_t
\]

The residual $\varepsilon_t$ is the spread. An ADF test is used as a residual
stationarity diagnostic, and an approximate half-life is estimated from a simple
AR(1)-style mean-reversion regression.


In [ ]:
def calculate_half_life(spread):
    spread = spread.dropna()

    lagged = spread.shift(1)
    delta = spread - lagged

    regression_data = pd.concat([delta, lagged], axis=1).dropna()
    regression_data.columns = ["Delta_Spread", "Lagged_Spread"]

    X = sm.add_constant(regression_data["Lagged_Spread"])
    model = sm.OLS(regression_data["Delta_Spread"], X).fit()

    lam = model.params["Lagged_Spread"]

    if lam < 0:
        return -np.log(2) / lam
    return np.inf


diagnostics = []

for _, row in raw_candidates.iterrows():
    asset_a = row["Asset_A"]
    asset_b = row["Asset_B"]

    y = train[asset_a]
    x = train[asset_b]

    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit()

    alpha = model.params.iloc[0]
    beta = model.params.iloc[1]
    spread = y - (alpha + beta * x)

    adf = adfuller(spread.dropna(), autolag="AIC")
    half_life = calculate_half_life(spread)

    diagnostics.append({
        "Pair": row["Pair"],
        "Cointegration_P": row["P_Value"],
        "Alpha": alpha,
        "Beta": beta,
        "ADF_P_Value": adf[1],
        "Half_Life_Days": half_life
    })

candidate_summary = (
    pd.DataFrame(diagnostics)
    .merge(
        cointegration_results[
            ["Pair", "FDR_P_Value", "Pass_FDR_5pct"]
        ],
        on="Pair"
    )
    .sort_values("Cointegration_P")
    .reset_index(drop=True)
)

candidate_summary.round(4)


## 5. Training-only pair selection

A simple predefined exploratory rule is used:

1. Raw Engle–Granger p-value < 0.05
2. Positive OLS hedge ratio
3. Finite, positive half-life
4. Choose the lowest raw cointegration p-value

This mechanically selects **TXN–GS** using training data only.


In [ ]:
eligible = candidate_summary[
    (candidate_summary["Cointegration_P"] < 0.05) &
    (candidate_summary["Beta"] > 0) &
    np.isfinite(candidate_summary["Half_Life_Days"]) &
    (candidate_summary["Half_Life_Days"] > 0)
].copy()

selected = eligible.sort_values("Cointegration_P").iloc[0]

asset_a, asset_b = selected["Pair"].split("-")

y_train = train[asset_a]
x_train = train[asset_b]

X_train = sm.add_constant(x_train)
training_model = sm.OLS(y_train, X_train).fit()

train_alpha = training_model.params.iloc[0]
train_beta = training_model.params.iloc[1]

estimated_half_life = selected["Half_Life_Days"]
z_window = max(10, int(round(estimated_half_life)))

print("Selected pair:", selected["Pair"])
print(f"Training cointegration p-value: {selected['Cointegration_P']:.6f}")
print(f"Training ADF p-value: {selected['ADF_P_Value']:.6f}")
print(f"Training alpha: {train_alpha:.4f}")
print(f"Training beta: {train_beta:.4f}")
print(f"Half-life: {estimated_half_life:.2f} trading days")
print("Z-score lookback:", z_window)
print("Passes 5% FDR correction:", selected["Pass_FDR_5pct"])


In [ ]:
train_spread = train[asset_a] - (
    train_alpha + train_beta * train[asset_b]
)

plt.figure(figsize=(12, 5))
plt.plot(train_spread, label=f"{asset_a}-{asset_b} training spread")
plt.axhline(train_spread.mean(), linestyle="--", label="Training mean")
plt.title(f"{asset_a}-{asset_b}: Training-Period Residual Spread")
plt.xlabel("Date")
plt.ylabel("Spread")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Trading rule

The final rule is frozen before the 2025 out-of-sample test:

- **Long spread:** Z < -2.0
- **Short spread:** Z > +2.0
- **Exit:** |Z| <= 0.5
- **Execution:** one trading day after the signal
- **Transaction cost:** 10 bps per unit of position turnover
- **Hedge ratio:** fixed from 2021–2023 training

The daily hedged spread return is based on the dollar P&L

\[
\Delta A_t - \beta \Delta B_t
\]

divided by previous-day gross capital

\[
A_{t-1}+|\beta|B_{t-1}.
\]


In [ ]:
ENTRY = 2.0
EXIT = 0.5
COST_RATE = 0.001

def generate_positions(z_score, entry=ENTRY, exit_threshold=EXIT):
    position = pd.Series(0.0, index=z_score.index)
    current = 0

    for date in z_score.index:
        z = z_score.loc[date]

        if current == 0:
            if z < -entry:
                current = 1
            elif z > entry:
                current = -1
        else:
            if abs(z) <= exit_threshold:
                current = 0

        position.loc[date] = current

    return position

def performance_metrics(net_return, gross_return, executed_position, cost):
    gross_equity = (1 + gross_return).cumprod()
    net_equity = (1 + net_return).cumprod()

    vol = net_return.std() * np.sqrt(252)

    sharpe = (
        net_return.mean() / net_return.std() * np.sqrt(252)
        if net_return.std() != 0 else np.nan
    )

    negative = net_return[net_return < 0]
    downside = negative.std() * np.sqrt(252)

    sortino = (
        net_return.mean() * 252 / downside
        if len(negative) > 1 and downside != 0 else np.nan
    )

    drawdown = net_equity / net_equity.cummax() - 1

    entries = (
        (executed_position != 0) &
        (executed_position.shift(1).fillna(0) == 0)
    ).sum()

    exits = (
        (executed_position == 0) &
        (executed_position.shift(1).fillna(0) != 0)
    ).sum()

    return {
        "Gross Return": gross_equity.iloc[-1] - 1,
        "Net Return": net_equity.iloc[-1] - 1,
        "Annualized Volatility": vol,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "Max Drawdown": drawdown.min(),
        "Entries": int(entries),
        "Exits": int(exits),
        "Transaction Cost": cost.sum()
    }, gross_equity, net_equity, drawdown


## 7. 2024 validation

The OLS alpha and beta remain frozen from training. Rolling Z-scores are calculated
using continuous historical spread data so the first validation observations have
sufficient lookback history.


In [ ]:
train_validation = pd.concat([train, validation])

tv_spread = train_validation[asset_a] - (
    train_alpha + train_beta * train_validation[asset_b]
)

tv_mean = tv_spread.rolling(z_window).mean()
tv_std = tv_spread.rolling(z_window).std()
tv_z = (tv_spread - tv_mean) / tv_std

validation_z = tv_z.loc[validation.index]
validation_position = generate_positions(validation_z)

validation_executed = validation_position.shift(1).fillna(0)

validation_pnl = (
    validation[asset_a].diff()
    - train_beta * validation[asset_b].diff()
)

validation_capital = (
    validation[asset_a].shift(1)
    + abs(train_beta) * validation[asset_b].shift(1)
)

validation_spread_return = (
    validation_pnl / validation_capital
).fillna(0)

validation_gross_return = (
    validation_executed * validation_spread_return
).fillna(0)

validation_turnover = (
    validation_executed.diff().abs()
    .fillna(validation_executed.abs())
)

validation_cost = validation_turnover * COST_RATE
validation_net_return = validation_gross_return - validation_cost

validation_metrics, validation_gross_equity, validation_net_equity, validation_drawdown = (
    performance_metrics(
        validation_net_return,
        validation_gross_return,
        validation_executed,
        validation_cost
    )
)

pd.Series(validation_metrics, name="2024 Validation")


### Validation result

The original run produced approximately **+6.41% net return**, **0.818 Sharpe**
and **-3.70% maximum drawdown** in 2024. No strategy parameter is retuned using
the final 2025 test.


## 8. Final 2025 out-of-sample test

The 2025 period is evaluated with the same training-period alpha, beta, half-life
lookback, entry/exit thresholds and transaction-cost assumption.


In [ ]:
full_history = pd.concat([train, validation, test])

full_spread = full_history[asset_a] - (
    train_alpha + train_beta * full_history[asset_b]
)

full_mean = full_spread.rolling(z_window).mean()
full_std = full_spread.rolling(z_window).std()
full_z = (full_spread - full_mean) / full_std

test_z = full_z.loc[test.index]
test_position = generate_positions(test_z)

# One-day execution lag
test_executed = test_position.shift(1).fillna(0)

# Hedge-adjusted spread return using frozen training beta
full_pnl = (
    full_history[asset_a].diff()
    - train_beta * full_history[asset_b].diff()
)

full_capital = (
    full_history[asset_a].shift(1)
    + abs(train_beta) * full_history[asset_b].shift(1)
)

full_spread_return = full_pnl / full_capital
test_spread_return = full_spread_return.loc[test.index].fillna(0)

test_gross_return = (
    test_executed * test_spread_return
).fillna(0)

test_turnover = (
    test_executed.diff().abs()
    .fillna(test_executed.abs())
)

test_cost = test_turnover * COST_RATE
test_net_return = test_gross_return - test_cost

test_metrics, test_gross_equity, test_net_equity, test_drawdown = (
    performance_metrics(
        test_net_return,
        test_gross_return,
        test_executed,
        test_cost
    )
)

pd.Series(test_metrics, name="2025 Out-of-Sample")


### Final out-of-sample result

The original run produced:

| Metric | 2025 OOS |
|---|---:|
| Gross return | -1.30% |
| Net return | -4.22% |
| Annualized volatility | 11.60% |
| Sharpe ratio | -0.317 |
| Sortino ratio | -0.304 |
| Maximum drawdown | -9.92% |
| Entries / exits | 15 / 15 |
| Modeled transaction cost | 3.00% |

The weak 2025 result is retained rather than optimized away. It is evidence that
the validation-period relationship did not generalize reliably.


## 9. Visual diagnostics


In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(test_gross_equity.index, test_gross_equity, label="Gross strategy")
plt.plot(test_net_equity.index, test_net_equity, label="Net strategy")
plt.axhline(1, linestyle="--", linewidth=1)
plt.title("TXN-GS Statistical Arbitrage Strategy — 2025 OOS Equity Curve")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(test_drawdown.index, test_drawdown * 100)
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("TXN-GS Statistical Arbitrage Strategy — 2025 OOS Drawdown")
plt.xlabel("Date")
plt.ylabel("Drawdown (%)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
long_entries = (
    (test_position == 1) &
    (test_position.shift(1).fillna(0) == 0)
)

short_entries = (
    (test_position == -1) &
    (test_position.shift(1).fillna(0) == 0)
)

exits = (
    (test_position == 0) &
    (test_position.shift(1).fillna(0) != 0)
)

plt.figure(figsize=(14, 6))
plt.plot(test_z.index, test_z, label="Z-score", linewidth=1.5)
plt.axhline(2.0, linestyle="--", label="+2 entry")
plt.axhline(-2.0, linestyle="--", label="-2 entry")
plt.axhline(0.5, linestyle=":", label="+0.5 exit")
plt.axhline(-0.5, linestyle=":", label="-0.5 exit")
plt.axhline(0, linewidth=1)

plt.scatter(test_z.index[long_entries], test_z[long_entries],
            marker="^", s=80, label="Long entry")
plt.scatter(test_z.index[short_entries], test_z[short_entries],
            marker="v", s=80, label="Short entry")
plt.scatter(test_z.index[exits], test_z[exits],
            marker="x", s=70, label="Exit")

plt.title("TXN-GS Statistical Arbitrage Strategy — 2025 OOS Trading Signals")
plt.xlabel("Date")
plt.ylabel("Z-score")
plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
validation_curve = validation_net_equity.reset_index(drop=True)
test_curve = test_net_equity.reset_index(drop=True)

plt.figure(figsize=(12, 6))
plt.plot(validation_curve.index, validation_curve, label="2024 validation")
plt.plot(test_curve.index, test_curve, label="2025 out-of-sample")
plt.axhline(1, linestyle="--", linewidth=1)
plt.title("Validation vs Out-of-Sample Performance")
plt.xlabel("Trading day")
plt.ylabel("Growth of $1")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 10. Interpretation and limitations

The strategy's 2024 validation performance did not persist in 2025. Gross 2025
performance was already slightly negative, and modeled turnover costs pushed the
net result further down. This makes the project more useful as a demonstration of
research discipline than as evidence of a profitable trading rule.

Important limitations:

- No pair survived the 5% Benjamini–Hochberg FDR correction.
- The selected pair is therefore exploratory and exposed to multiple-testing risk.
- The hedge ratio is static and estimated only on 2021–2023 data.
- Costs are simplified and omit bid-ask spread, slippage, borrow cost and market impact.
- Adjusted-close data are an approximation for an executable institutional dataset.
- The final out-of-sample period covers only one year.
- The universe was expanded during research after an initial smaller-universe prototype;
  the 2025 test was nevertheless left untouched after the final specification was frozen.

**Conclusion:** the pipeline demonstrates cointegration screening, residual diagnostics,
lagged execution, cost-aware backtesting and chronological model evaluation. The main
finding is model instability rather than persistent profitability.
